# Fine-tune masked-reconstruction-pretrained ResNet-18, ps32 SCV

This notebook fine-tunes a binary ResNet-18 classifier initialized from the masked reconstruction SSL encoder checkpoint. It uses the same ps32 5-cluster spatial cross-validation protocol as the scratch baseline.

## 1. Purpose and experiment configuration

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

model_name = "masked_recon_resnet18"
pretraining = "masked_recon"
patch_size = 32
patch_tag = "ps32"
random_seed = 42
val_fraction = 0.2
batch_size = 16
encoder_learning_rate = 1e-5
head_learning_rate = 1e-4
weight_decay = 1e-4
max_epochs = 100
early_stopping_patience = 15
gradient_clip_norm = 5.0
dropout = 0.4
scratch_mean_auc = 0.778842
scratch_std_auc = 0.104754

patch_index_csv = PROJECT_ROOT / "data/processed/patches/labeled_patch_index_ps32_common_balanced.csv"
raster_dir = PROJECT_ROOT / "data/processed/rasters_cleaned"
encoder_checkpoint_path = PROJECT_ROOT / "checkpoints/ssl_pretrained/masked_recon/resnet18_masked_recon_ps32_encoder_best.pt"
output_root = PROJECT_ROOT / "outputs/R1_ssl_task_comparison/masked_recon"
figure_root = PROJECT_ROOT / "figures/R1_ssl_task_comparison/masked_recon"
checkpoint_dir = PROJECT_ROOT / "checkpoints/finetuned/masked_recon_resnet18"

## 2. Import packages and project modules

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

from src.metrics import compute_binary_metrics, confusion_matrix_metrics, find_best_f1_threshold
from src.models_resnet18 import create_resnet18_binary_classifier, load_ssl_encoder_weights_into_classifier
from src.patch_dataset import RasterPatchDataset, list_raster_files
from src.plotting import plot_pr_curves_all_folds, plot_roc_curves_all_folds, plot_training_curves
from src.train_finetune import evaluate_model, train_scv_fold
from src.utils import count_trainable_parameters, ensure_dir, get_device, load_checkpoint, set_global_seed

## 3. Set random seeds and device

In [ ]:
set_global_seed(random_seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = torch.cuda.is_available()
num_workers = 0
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"CUDA GPU: {torch.cuda.get_device_name(0)}")
    print(f"torch.version.cuda: {torch.version.cuda}")
else:
    print("WARNING: CUDA is not available; training will run on CPU.")

## 4. Load patch index, cleaned rasters, and pretrained encoder checkpoint

In [ ]:
patch_index = pd.read_csv(patch_index_csv)
raster_files = list_raster_files(raster_dir)
encoder_checkpoint = torch.load(encoder_checkpoint_path, map_location="cpu")
print(f"Patch index: {patch_index_csv}")
print(f"Pretrained encoder checkpoint: {encoder_checkpoint_path}")
print(f"SSL checkpoint epoch: {encoder_checkpoint.get('epoch')}")
print(f"SSL checkpoint val_loss: {encoder_checkpoint.get('val_loss')}")

## 5. Verify input data quality

In [ ]:
assert patch_index["valid_patch"].astype(bool).all()
assert np.isclose(patch_index["nodata_ratio"].to_numpy(float), 0.0).all()
print(f"total sample count: {len(patch_index)}")
print("label counts:")
print(patch_index["label"].value_counts().sort_index().to_string())
print("cluster_id x label table:")
print(pd.crosstab(patch_index["cluster_id"], patch_index["label"]).to_string())
print(f"patch_size: {patch_size}")
print(f"number of raster channels: {len(raster_files)}")
print(f"dataset path: {patch_index_csv}")
print(f"pretrained weights loaded successfully: checkpoint readable")
print(f"batch size: {batch_size}")
print(f"encoder learning rate: {encoder_learning_rate}")
print(f"head learning rate: {head_learning_rate}")
print(f"max epochs: {max_epochs}")
print(f"early stopping patience: {early_stopping_patience}")

## 6. Define modified ResNet-18 classifier and load SSL encoder weights

In [ ]:
preview_model = create_resnet18_binary_classifier(in_channels=14, dropout=dropout, small_patch_stem=True, pretrained=False)
preview_load_info = load_ssl_encoder_weights_into_classifier(preview_model, encoder_checkpoint_path, strict_encoder=True)
print(f"preview loaded pretrained encoder keys: {len(preview_load_info['loaded_keys'])}")
print(f"model parameter count: {count_trainable_parameters(preview_model)}")
del preview_model

## 7. Define SCV fold loop

Fold loop follows cluster_id 0..4 as held-out test clusters, matching the scratch baseline.

## 8. Compute fold-specific channel normalization from training data only

For fair comparison with the scratch baseline, supervised fine-tuning normalization is computed from training samples within each fold, not from SSL pretraining statistics.

## 9. Fine-tune fold-specific masked-recon-pretrained ResNet-18 models

The executable training loop is run by the project workflow script using the helpers imported above.

## 10. Evaluate fold-specific test performance

Best-F1 thresholds are selected from validation predictions only and then applied to the held-out test cluster.

## 11. Save predictions, metrics, logs, and checkpoints

Outputs are written to the R1 masked reconstruction comparison folders.

## 12. Plot ROC, PR, and training curves

ROC, PR, and training curves are saved as PNG figures.

## 13. Compare summary metrics against scratch baseline

Scratch reference: mean_auc = 0.778842, std_auc = 0.104754.

## 14. Print final summary and next-step notes

This fine-tuned masked-reconstruction model is the first SSL-pretrained comparison against the scratch ResNet-18 ps32 baseline.